In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load datasets
temperature_df = pd.read_csv("/content/drive/MyDrive/Dataset/tbltemperature.csv")  # Temperature data
neutrophils_df = pd.read_csv("/content/drive/MyDrive/Dataset/tblwbc.csv")           # Neutrophil counts
stool_df = pd.read_csv("/content/drive/MyDrive/Dataset/tblASVsamples.csv")         # Stool consistency
allo_hct = pd.read_csv("/content/drive/MyDrive/Dataset/tblhctmeta.csv")

In [ ]:
temperature_df.info()

In [ ]:
neutrophils_df.info()

In [ ]:
stool_df.info()

In [ ]:
allo_hct.info()

In [ ]:
# Convert 'PatientID' to string in all relevant DataFrames
temperature_df['PatientID'] = temperature_df['PatientID'].astype(str)
neutrophils_df['PatientID'] = neutrophils_df['PatientID'].astype(str)
stool_df['PatientID'] = stool_df['PatientID'].astype(str)
allo_hct['PatientID'] = allo_hct['PatientID'].astype(str)

In [ ]:
# Filter relevant columns
temperature_df = temperature_df[['PatientID', 'DayRelativeToNearestHCT', 'MaxTemperature']]
neutrophils_df = neutrophils_df[neutrophils_df['BloodCellType'] == 'Neutrophils'][['PatientID', 'DayRelativeToNearestHCT', 'Value']]
stool_df = stool_df[['PatientID', 'DayRelativeToNearestHCT', 'Consistency']]
allo_hct = allo_hct[['PatientID', 'TimepointOfTransplant', 'Disease']]

In [ ]:
 # Rename columns for consistency
neutrophils_df.rename(columns={'Value': 'NeutrophilCount'}, inplace=True)

In [ ]:
# Merge datasets on PatientID and DayRelativeToNearestHCT
merged_df = pd.merge(allo_hct, temperature_df, on=['PatientID'], how='outer')
merged_df = pd.merge(merged_df, neutrophils_df, on=['PatientID', 'DayRelativeToNearestHCT'], how='outer')
merged_df = pd.merge(merged_df, stool_df, on=['PatientID', 'DayRelativeToNearestHCT'], how='outer')

In [ ]:
merged_df.head(5)

In [ ]:
# Create a uniform timeline
min_day = -15  # Minimum day relative to allo-HSCT
max_day = 35   # Maximum day relative to allo-HSCT

In [ ]:
# Generate a complete timeline for each patient
all_days = pd.DataFrame(
    {'DayRelativeToNearestHCT': range(min_day, max_day + 1)})
patients = merged_df['PatientID'].unique()

In [ ]:
# Get the total number of patients
num_patients = patients.shape[0]  # or len(patients)
print(f"Total number of patients: {num_patients}")

In [ ]:
# Initialize final dataset
aligned_data = []

In [ ]:
for patient in patients:
    # Extract patient-specific data
    patient_data = merged_df[merged_df['PatientID'] == patient]

    # Merge with all_days to ensure a complete timeline
    patient_timeline = all_days.merge(
        patient_data, on='DayRelativeToNearestHCT', how='left')
    patient_timeline['PatientID'] = patient  # Add PatientID column
    aligned_data.append(patient_timeline)

In [ ]:
# Combine all patient timelines
final_df = pd.concat(aligned_data, ignore_index=True)

In [ ]:
# Handle missing data
# Forward-fill temperature
final_df['MaxTemperature'] = final_df['MaxTemperature'].ffill()
# Forward-fill neutrophil counts
final_df['NeutrophilCount'] = final_df['NeutrophilCount'].ffill()
# Fill missing stool consistency as 'Unknown'
final_df['Consistency'] = final_df['Consistency'].fillna('Unknown')

In [ ]:
final_df.shape

In [ ]:
# Save to a CSV for modeling
final_df.to_csv("processed_dataset.csv", index=False)
print("Dataset processed and saved as 'processed_dataset.csv'")